### 1. Step. Airport data from Aerodatabox

We will use the [AeroDataBox API](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/), which can retrieve all sorts of information about flights and airports.

In [21]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

In [41]:
load_dotenv()
rapidapi_key = os.getenv("FLIGHT_KEY")
connection_string = os.getenv("CON_STRING")

In [40]:
rapidapi_key

In [ ]:
744e507467mshb0bda2b1c5b7793p1e3672jsn964936934962

We'll simulate a DataFrame like the kind hosted on our sample database. You should use the table on your own database to find airports for those cities.

In [32]:
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

,city_id,name,country,latitude,longitude
0,1,Berlin,Germany,52.5200,13.405
1,2,Hamburg,Germany,53.5500,10.000
2,3,Munich,Germany,48.1375,11.575


In [33]:
def get_airports(cities_df):
    # API headers
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
    

    # DataFrame to store results
    all_airports = []

    for _, row in cities_df.iterrows():
    # Construct the URL with the latitude and longitude
        querystring = {"lat":row["latitude"],"lon":row["longitude"],"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            all_airports.append(airports)
        else:
            print(f"WARNING: Failed to retrieve airports for {row['name']}")

    return pd.concat(all_airports, ignore_index=True)

In [34]:
get_airports(cities_df)

ValueError: No objects to concatenate

In [29]:
get_airports(cities_df).to_sql(
    "airports",
    con=connection_string,
    if_exists="replace",
    index=False
)

ValueError: No objects to concatenate

### ### 1. Step. Arrival data from Aerodatabox

### 
_____

We can start by loading our needed secrets and reading the `'cities'` table from Gans's database.

In [17]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

In [26]:
load_dotenv()
connection_string = os.getenv("CON_STRING")
rapidapi_key = os.getenv("RAPIDAPI_KEY")
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

,city_id,name,country,latitude,longitude
0,1,Berlin,Germany,52.5200,13.405
1,2,Hamburg,Germany,53.5500,10.000
2,3,Munich,Germany,48.1375,11.575


### Step 1: Airports

A function for finding airports is already provided. We'll modify it just a little, selecting only a few columns from each result and renaming one of them.

In [27]:
def get_airports(cities_df):
    # API constants
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

    # DataFrame to store results
    all_airports = []

    for _, row in cities_df.iterrows():
        querystring = {"lat":row["latitude"],"lon":row["longitude"],"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            airports["city_id"] = row["city_id"] # add city_id for foreign key reference
            all_airports.append(airports)
        else:
            print(f"WARNING: Failed to retrieve airports for {row['name']}")

    airports_df = pd.concat(all_airports, ignore_index=True) # make one DataFrame from individual results
    airports_df = airports_df[["icao", "name", "city_id"]] 

    return airports_df

In [28]:
airports_df = get_airports(cities_df)
airports_df

ValueError: No objects to concatenate

This result looks just fine. Before proceeding with flights, let's write a table definition to `03_gans_schema.sql` and populate the table with `airports_df`. One thing worth noting, we'll add an extra column for `'active'` to the SQL table. This way if AeroDataBox provides a bad airport (like Tegel) we have a way to exclude it from flight searches. Equally valid, we could work out a method to blacklist certain airports from `airports_df`.

In [ ]:
airports_df.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

### Step 2: Flights

Let's begin by reading from the database's `'airports'` table. Here, we select only airports that we've flagged as "active".

In [38]:
airports_df = pd.read_sql("airports" , con=connection_string)
airports_df

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
1,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229
2,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100


#### One airport, one time window

Following the approach taken with weather forecasts, we'll start small with a single call to the API and then add layers of loops to retrieve more and more data.

In [42]:
# copy headers from above
headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

# copy url and querystring from AeroDataBox website, filling in one icao code
url = "https://aerodatabox.p.rapidapi.com/flights/airports/icao/EDDB/2026-08-22T00:00/2026-08-22T11:59"
querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}

response = requests.get(url, headers=headers, params=querystring)
print(response.json())

{'arrivals': [{'movement': {'airport': {'icao': 'LTFM', 'iata': 'IST', 'name': 'Istanbul', 'location': {'lat': 41.275276, 'lon': 28.751944}, 'countryCode': 'tr', 'timeZone': 'Europe/Istanbul'}, 'scheduledTime': {'utc': '2026-08-22 03:40Z', 'local': '2026-08-22 05:40+02:00'}, 'revisedTime': {'utc': '2026-08-22 03:26Z', 'local': '2026-08-22 05:26+02:00'}, 'runwayTime': {'utc': '2026-08-22 03:26Z', 'local': '2026-08-22 05:26+02:00'}, 'terminal': '1', 'gate': 'X10', 'baggageBelt': 'A1', 'runway': '25L', 'quality': ['Basic', 'Live']}, 'number': 'TK 1220', 'callSign': 'THY9XT', 'status': 'Arrived', 'codeshareStatus': 'IsOperator', 'isCargo': False, 'aircraft': {'reg': 'TC-LPC', 'modeS': '4BB203', 'model': 'Airbus A321 NEO'}, 'airline': {'name': 'Turkish', 'iata': 'TK', 'icao': 'THY'}}, {'movement': {'airport': {'icao': 'LTAI', 'iata': 'AYT', 'name': 'Antalya', 'location': {'lat': 36.8987, 'lon': 30.8005}, 'countryCode': 'tr', 'timeZone': 'Europe/Istanbul'}, 'scheduledTime': {'utc': '2026-08-

We can now begin to explore the response

In [43]:
response_json = response.json()
response_json.keys()

dict_keys(['arrivals'])

In [44]:
# looks like a list
response_json["arrivals"]

[{'movement': {'airport': {'icao': 'LTFM',
    'iata': 'IST',
    'name': 'Istanbul',
    'location': {'lat': 41.275276, 'lon': 28.751944},
    'countryCode': 'tr',
    'timeZone': 'Europe/Istanbul'},
   'scheduledTime': {'utc': '2026-08-22 03:40Z',
    'local': '2026-08-22 05:40+02:00'},
   'revisedTime': {'utc': '2026-08-22 03:26Z',
    'local': '2026-08-22 05:26+02:00'},
   'runwayTime': {'utc': '2026-08-22 03:26Z',
    'local': '2026-08-22 05:26+02:00'},
   'terminal': '1',
   'gate': 'X10',
   'baggageBelt': 'A1',
   'runway': '25L',
   'quality': ['Basic', 'Live']},
  'number': 'TK 1220',
  'callSign': 'THY9XT',
  'status': 'Arrived',
  'codeshareStatus': 'IsOperator',
  'isCargo': False,
  'aircraft': {'reg': 'TC-LPC', 'modeS': '4BB203', 'model': 'Airbus A321 NEO'},
  'airline': {'name': 'Turkish', 'iata': 'TK', 'icao': 'THY'}},
 {'movement': {'airport': {'icao': 'LTAI',
    'iata': 'AYT',
    'name': 'Antalya',
    'location': {'lat': 36.8987, 'lon': 30.8005},
    'countryCode'

In [45]:
# try to understand one arrival
# check the webpage as well, there may be items not present in every arrival!
arrival = response_json["arrivals"][0]
arrival

{'movement': {'airport': {'icao': 'LTFM',
   'iata': 'IST',
   'name': 'Istanbul',
   'location': {'lat': 41.275276, 'lon': 28.751944},
   'countryCode': 'tr',
   'timeZone': 'Europe/Istanbul'},
  'scheduledTime': {'utc': '2026-08-22 03:40Z',
   'local': '2026-08-22 05:40+02:00'},
  'revisedTime': {'utc': '2026-08-22 03:26Z',
   'local': '2026-08-22 05:26+02:00'},
  'runwayTime': {'utc': '2026-08-22 03:26Z',
   'local': '2026-08-22 05:26+02:00'},
  'terminal': '1',
  'gate': 'X10',
  'baggageBelt': 'A1',
  'runway': '25L',
  'quality': ['Basic', 'Live']},
 'number': 'TK 1220',
 'callSign': 'THY9XT',
 'status': 'Arrived',
 'codeshareStatus': 'IsOperator',
 'isCargo': False,
 'aircraft': {'reg': 'TC-LPC', 'modeS': '4BB203', 'model': 'Airbus A321 NEO'},
 'airline': {'name': 'Turkish', 'iata': 'TK', 'icao': 'THY'}}

In [46]:
arrival.keys()

dict_keys(['movement', 'number', 'callSign', 'status', 'codeshareStatus', 'isCargo', 'aircraft', 'airline'])

In [47]:
arrival_dict = {
    "depart_airport": arrival["movement"]["airport"]["name"],
    "depart_country": arrival["movement"]["airport"]["countryCode"].upper(), # country codes are more often all capital letters
    "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
    "arrive_time_revised": arrival["movement"]["revisedTime"]["local"],
    "flight_number": arrival["number"],
    "aircraft": arrival["aircraft"]["model"]
}
arrival_dict

{'depart_airport': 'Istanbul',
 'depart_country': 'TR',
 'arrive_time_scheduled': '2026-08-22 05:40+02:00',
 'arrive_time_revised': '2026-08-22 05:26+02:00',
 'flight_number': 'TK 1220',
 'aircraft': 'Airbus A321 NEO'}

Now let's see if this works with the rest of the flights in the response.

In [ ]:
arrivals = [] # empty list to store arrivals
for arrival in response_json["arrivals"]:
    arrival_dict = {
        "depart_airport": arrival["movement"]["airport"]["name"],
        "depart_country": arrival["movement"]["airport"]["countryCode"].upper(),
        "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
        "arrive_time_revised": arrival["movement"]["revisedTime"]["local"],
        "flight_number": arrival["number"],
        "aircraft": arrival["aircraft"]["model"]
    }
    arrivals.append(arrival_dict)

flights_df = pd.DataFrame(arrivals)

Looks like we get the same problem we saw with weather: not every key is included in every arrival. `.get()` will help us here again.

In [52]:
arrivals = [] # empty list to store arrivals
for arrival in response_json["arrivals"]:
    arrival_dict = {
        "depart_airport": arrival["movement"]["airport"]["name"],
        "depart_country": arrival["movement"]["airport"]["countryCode"].upper(),
        "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
        "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
        "flight_number": arrival["number"],
        "aircraft": arrival["aircraft"].get("model", None)
    }
    arrivals.append(arrival_dict)

flights_df = pd.DataFrame(arrivals)
flights_df

,depart_airport,depart_country,arrive_time_scheduled,arrive_time_revised,flight_number,aircraft
0,Istanbul,TR,2026-08-22 05:40+02:00,2026-08-22 05:26+02:00,TK 1220,Airbus A321 NEO
1,Antalya,TR,2026-08-22 06:05+02:00,2026-08-22 05:27+02:00,XQ 668,Boeing 737-800
2,İzmir,TR,2026-08-22 06:05+02:00,2026-08-22 05:48+02:00,XQ 966,Boeing 737-800
3,Adana,TR,2026-08-22 06:45+02:00,2026-08-22 06:20+02:00,XQ 1774,Boeing 737 MAX 8
4,Beirut,LB,2026-08-22 06:10+02:00,2026-08-22 06:30+02:00,SR 1599,Airbus A320-200
...,...,...,...,...,...,...
75,London,GB,2026-08-22 11:45+02:00,2026-08-22 11:37+02:00,EW 8461,Airbus A320 NEO
76,Tirana,AL,2026-08-22 11:30+02:00,2026-08-22 11:39+02:00,W4 5105,Airbus A321
77,Palma De Mallorca,ES,2026-08-22 12:25+02:00,2026-08-22 11:51+02:00,U2 5120,Airbus A320
78,Beirut,LB,2026-08-22 12:05+02:00,2026-08-22 11:54+02:00,ME 245,Airbus A320


Our API will only let us retrieve flight data for up to 12 hours at a time. To cover all of tomorrow, we'll need to build a `for` loop. We'll also need to work out how to get tomorrow's date.

#### One airport, two time windows

In [53]:
today = pd.Timestamp.now().date()
one_day = pd.Timedelta(1, "day")
tomorrow = today+one_day
tomorrow_str = tomorrow.strftime("%Y-%m-%d")
tomorrow_str

'2026-10-24'

In [58]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]
airport = airports_df.loc[0] # use the DataFrame this time, instead of copying call details from website

arrivals = [] # empty list to store arrivals
for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport['icao']}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
    querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
    response = requests.get(url, headers=headers, params=querystring)

    if response.status_code == 200:
        data = response.json()["arrivals"]
        for arrival in data:
            arrival_dict = {
                "arrive_icao": airport["icao"], # add foreign-key information
                "depart_icao": arrival["movement"]["airport"].get("icao", None), # add this since countryCode and maybe name aren't reliable
                "depart_airport": arrival["movement"]["airport"].get("name", None),
                "depart_country": arrival["movement"]["airport"].get("countryCode", "XX").upper(),
                "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                "flight_number": arrival.get("number", None),
                "aircraft": arrival.get("aircraft", {}).get("model", None)
            }
            arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df

,arrive_icao,depart_icao,depart_airport,depart_country,arrive_time_scheduled,arrive_time_revised,flight_number,aircraft
0,EDDB,LTBJ,İzmir,TR,2026-10-24 06:05+02:00,2026-10-24 06:05+02:00,XQ 966,Boeing 737-700 (winglets)
1,EDDB,OLBA,Beirut,LB,2026-10-24 06:10+02:00,2026-10-24 06:10+02:00,SR 1599,Airbus A320
2,EDDB,LTDB,Adana,TR,2026-10-24 06:45+02:00,2026-10-24 06:45+02:00,XQ 1774,Boeing 737-700 (winglets)
3,EDDB,LTAI,Antalya,TR,2026-10-24 06:50+02:00,2026-10-24 06:50+02:00,TK 3350,737 MAX 9
4,EDDB,LROP,Bucharest,RO,2026-10-24 07:00+02:00,2026-10-24 07:00+02:00,W4 3109,Airbus A321-200
...,...,...,...,...,...,...,...,...
246,EDDB,EDDF,Frankfurt-am-Main,DE,2026-10-24 22:55+02:00,2026-10-24 22:55+02:00,LH 206,Airbus A321 NEO
247,EDDB,EETN,Tallinn,EE,2026-10-24 22:55+02:00,2026-10-24 22:55+02:00,FR 2733,Boeing 737-800 (winglets)
248,EDDB,LEMG,Málaga,ES,2026-10-24 23:00+02:00,2026-10-24 23:00+02:00,EW 8537,Airbus A321-200
249,EDDB,EIDW,Dublin,IE,2026-10-24 23:00+02:00,2026-10-24 23:00+02:00,FR 5418,Boeing 737 MAX 8


#### All airports, two time windows

In [ ]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]

arrivals = [] # empty list to store arrivals
for _, airport in airports_df.iterrows():
    for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
        url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport['icao']}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
        querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
        response = requests.get(url, headers=headers, params=querystring)
    
        if response.status_code == 200:
            data = response.json()["arrivals"]
            for arrival in data:
                arrival_dict = {
                    "arrive_icao": airport["icao"], # add foreign-key information
                    "depart_icao": arrival["movement"]["airport"].get("icao", None), # add this since countryCode and maybe name aren't reliable
                    "depart_airport": arrival["movement"]["airport"].get("name", None),
                    "depart_country": arrival["movement"]["airport"].get("countryCode", None), # move .upper() to deal with NoneTypes
                    "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                    "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                    "flight_number": arrival.get("number", None),
                    "aircraft": arrival.get("aircraft", {}).get("model", None)
                }
                arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df["scheduled_arrival_time"] = flights_df["scheduled_arrival_time"].str[:-6]

flights_df["depart_country"] = flights_df["depart_country"].str.upper()
flights_df

,arrive_icao,depart_icao,depart_airport,depart_country,arrive_time_scheduled,arrive_time_revised,flight_number,aircraft
0,EDDB,LTBJ,İzmir,TR,2026-10-24 06:05+02:00,2026-10-24 06:05+02:00,XQ 966,Boeing 737-700 (winglets)
1,EDDB,OLBA,Beirut,LB,2026-10-24 06:10+02:00,2026-10-24 06:10+02:00,SR 1599,Airbus A320
2,EDDB,LTDB,Adana,TR,2026-10-24 06:45+02:00,2026-10-24 06:45+02:00,XQ 1774,Boeing 737-700 (winglets)
3,EDDB,LTAI,Antalya,TR,2026-10-24 06:50+02:00,2026-10-24 06:50+02:00,TK 3350,737 MAX 9
4,EDDB,LROP,Bucharest,RO,2026-10-24 07:00+02:00,2026-10-24 07:00+02:00,W4 3109,Airbus A321-200
...,...,...,...,...,...,...,...,...
858,EDDM,GCTS,Tenerife Island,ES,2026-10-24 23:10+02:00,2026-10-24 23:10+02:00,DE 1585,Airbus A320
859,EDDM,GCTS,Tenerife Island,ES,2026-10-24 23:15+02:00,2026-10-24 23:15+02:00,X3 2197,Boeing 737-800
860,EDDM,EGLL,London,GB,2026-10-24 23:20+02:00,2026-10-24 23:20+02:00,LH 2481,Airbus A320 NEO
861,EDDM,GCFV,Fuerteventura Island,ES,2026-10-24 23:25+02:00,2026-10-24 23:25+02:00,X3 2163,Boeing 737-800


Success! Now add a table definition to the database before pushing the data.

In [59]:
# we get "Incorrect datetime value: '2026-08-22 05:40+02:00' for column 'arrive_time_scheduled' at row 1", so let's convert to datetime
flights_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   arrive_icao            251 non-null    object
 1   depart_icao            251 non-null    object
 2   depart_airport         251 non-null    object
 3   depart_country         251 non-null    object
 4   arrive_time_scheduled  251 non-null    object
 5   arrive_time_revised    251 non-null    object
 6   flight_number          251 non-null    object
 7   aircraft               251 non-null    object
dtypes: object(8)
memory usage: 15.8+ KB


In [60]:
flights_df["arrive_time_scheduled"] = pd.to_datetime(flights_df["arrive_time_scheduled"])
flights_df["arrive_time_revised"] = pd.to_datetime(flights_df["arrive_time_revised"])

In [ ]:
flights_df.to_sql(
    "flights",
    con=connection_string,
    if_exists="append",
    index=False
)